In [30]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv026931/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [31]:
from pyspark.sql.functions import *

In [32]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header","true") \
.option("inferschema","true") \
.load("/user/itv026931/lendingclubproject/raw/loans_defaulters_csv")

In [4]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [33]:
loans_def_raw_schema = """member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,
inq_last_6mths float, total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"""

In [34]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header","true") \
.schema(loans_def_raw_schema) \
.load("/user/itv026931/lendingclubproject/raw/loans_defaulters_csv")

In [7]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [8]:
loans_def_raw_df

member_id,delinq_2yrs,delinq_amnt,pub_rec,pub_rec_bankruptcies,inq_last_6mths,total_rec_late_fee,mths_since_last_delinq,mths_since_last_record
3a366efe16c243e91...,0.0,0.0,1.0,1.0,0.0,0.0,null,83.0
9f3b33873233b9a42...,1.0,0.0,0.0,0.0,0.0,0.0,12.0,null
81d1bd5c408965373...,2.0,0.0,0.0,0.0,1.0,0.0,20.0,null
3ece6a3c9c20d41b9...,0.0,0.0,0.0,0.0,0.0,0.0,35.0,null
51686a0cbb9cfc4a4...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
34ee3c8e3bd6e25a4...,0.0,0.0,1.0,1.0,0.0,0.0,null,71.0
75ca7ec64ad50dc30...,0.0,0.0,0.0,0.0,0.0,0.0,74.0,null
c853bf125194e4998...,0.0,0.0,0.0,0.0,0.0,0.0,null,null
fc5399a74637cfeb9...,0.0,0.0,1.0,1.0,0.0,0.0,41.0,104.0
0982b107c47ae1c72...,0.0,0.0,0.0,0.0,0.0,0.0,33.0,null


In [35]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs",col("delinq_2yrs").cast("integer")).fillna(0,subset=["delinq_2yrs"])

In [14]:
loans_def_processed_df.filter("delinq_2yrs is null").count()

0

In [37]:
loans_def_delinq_temp = loans_def_processed_df.withColumn("mths_since_last_delinq",col("mths_since_last_delinq").cast("integer"))

In [38]:
loans_def_delinq_df = loans_def_delinq_temp.select("member_id","delinq_2yrs","delinq_amnt","mths_since_last_delinq").filter("delinq_2yrs > 0 OR mths_since_last_delinq > 0")

In [39]:
loans_def_records_enq_df = loans_def_processed_df.select("member_id").filter("pub_rec > 0.0 OR pub_rec_bankruptcies > 0.0 OR inq_last_6mths > 0.0")

In [18]:
loans_def_records_enq_df

member_id
3a366efe16c243e91...
81d1bd5c408965373...
34ee3c8e3bd6e25a4...
fc5399a74637cfeb9...
bd51a5ad2365bf9a7...
9e650623802202f39...
ab9503add192b5cb2...
aa80f60c06b7cbdd1...
eec30639c6487420f...
ae765f19d3f87daad...


In [40]:
loans_def_p_pub_rec_df = loans_def_raw_df.withColumn("pub_rec",col("pub_rec").cast("integer")).fillna(0,subset = ["pub_rec"])

In [41]:
loans_def_p_pub_rec_bankruptcies_df = loans_def_p_pub_rec_df.withColumn("pub_rec_bankruptcies",col("pub_rec_bankruptcies").cast("integer")).fillna(0,subset = ["pub_rec_bankruptcies"])

In [42]:
loans_def_p_inq_last_6mths_df = loans_def_p_pub_rec_bankruptcies_df.withColumn("inq_last_6mths",col("inq_last_6mths").cast("integer")).fillna(0,subset = ["inq_last_6mths"])

In [43]:
loans_def_p_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")

In [44]:
loans_def_detail_records_enq_df = loans_def_p_inq_last_6mths_df.select("member_id","pub_rec","pub_rec_bankruptcies","inq_last_6mths")

In [45]:
loans_def_detail_records_enq_df

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
3a366efe16c243e91...,1,1,0
9f3b33873233b9a42...,0,0,0
81d1bd5c408965373...,0,0,1
3ece6a3c9c20d41b9...,0,0,0
51686a0cbb9cfc4a4...,0,0,0
34ee3c8e3bd6e25a4...,1,1,0
75ca7ec64ad50dc30...,0,0,0
c853bf125194e4998...,0,0,0
fc5399a74637cfeb9...,1,1,0
0982b107c47ae1c72...,0,0,0


In [46]:
loans_def_detail_records_enq_df.write \
.option("header","true") \
.format("csv") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_def_detail_records_enq_df_csv") \
.save()

In [47]:
loans_def_detail_records_enq_df.write \
.option("header","true") \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026931/lendingclubproject/cleaned/loans_def_detail_records_enq_df_parquet") \
.save()